# Router Before the LLM: submission notebook

A cost-aware intent router for a personal finance assistant. This notebook is thin on purpose: it imports from `src/router/` and calls functions, so the same code runs here, in `scripts/` and in the demo.

**How to run:** Restart and Run All. With `FAST = True` (the default) it needs no GPU, no API key and no paid call. Numbers produced in fast mode are labelled as fast-mode numbers. The reported results are loaded from `results/`.

Sections follow the marking rubric: problem, data and preprocessing, models, evaluation, router and cost, limitations.

In [ ]:
# Cell 1: environment.
# On Colab: clone the repo and install the pinned packages. Locally: nothing to install.
import os
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # set once the GitHub repo exists (needed for the Colab clean-run check)

if IN_COLAB:
    import subprocess
    if not REPO_URL:
        raise RuntimeError("REPO_URL is not set yet, so the repo cannot be cloned on Colab.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "router-project"], check=True)
    os.chdir("router-project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    sys.path.insert(0, os.path.abspath("src"))  # make `import router` work without a restart

print("Python", sys.version.split()[0])
for pkg in ["torch", "transformers", "datasets", "sentence-transformers", "scikit-learn", "pandas"]:
    print(f"{pkg:>22} {version(pkg)}")

In [ ]:
# Cell 2: run mode and seeds.
FAST = True  # True: quick run, no GPU or API key. False: full five-seed reproduction (hours).
SEED = 42

import random

import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print("FAST =", FAST, "| SEED =", SEED)

## 1. Problem

A finance assistant sends every user query to an LLM, which costs money and time on each call. A small local model (the router) looks at each query first and either handles it directly, rejects it as out of scope, or escalates it to the LLM when unsure.

**Research question:** how many LLM calls can a small router safely remove, and at what error cost? The headline metric is coverage at matched risk: the largest share of queries the router can decide itself while its error rate stays at or below the LLM's.

## 2. Data and preprocessing

In-scope queries come from **Banking77** (Casanueva et al., 2020): 77 banking intents. Out-of-scope queries come from **CLINC150** (Larson et al., 2019). Both are loaded at a pinned Hub revision. Splits are created once by `scripts/make_splits.py` and saved to `data/processed/`, so every system sees the same data.

Pipeline (`src/router/preprocess.py`): unicode NFKC normalisation and whitespace cleanup, removal of empty rows and duplicates, a train/test leakage check, a stratified 10% validation split, token length checks and class balance. The full record is in `results/data_report.md`.

In [ ]:
import json

import pandas as pd

from router import data

splits = {name: data.load_split(name) for name in data.SPLIT_FILES}
pd.DataFrame({"rows": {name: len(df) for name, df in splits.items()}})

In [ ]:
# Rows removed at each step, per source (every removed row is listed in removed_rows.csv)
removed = pd.read_csv(data.PROCESSED_DIR / data.REMOVED_ROWS_FILE, keep_default_na=False)
removed.groupby(["source", "reason"]).size().unstack(fill_value=0)

In [ ]:
# Token lengths per tokeniser, measured on train. max_length is used by S1, S2 and S3.
with open(data.PROCESSED_DIR / "token_lengths.json") as f:
    lengths = json.load(f)
pd.DataFrame(
    {
        name: {
            "p50": v["single"]["p50"],
            "p99": v["single"]["p99"],
            "max": v["single"]["max"],
            "max_length": v["single"]["max_length"],
            "test truncated": f"{v['truncated_share']['test']:.2%}",
            "pair max_length (S3)": v["pair"]["max_length"],
        }
        for name, v in lengths.items()
    }
).T

In [ ]:
# Class balance: queries per intent in train. Uneven counts are why macro averages are primary.
import matplotlib.pyplot as plt

counts = splits["train"]["label"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(counts)), counts.values)
ax.set_xlabel("intent (sorted by count)")
ax.set_ylabel("train queries")
ax.set_title(f"Banking77 train: {counts.min()} to {counts.max()} queries per intent")
plt.show()

## 3. Models

Added in M1 (S0 TF-IDF + logistic regression, S1 fine-tuned encoder) and M2 (S2 embedding nearest neighbour, S3 pairwise paraphrase matcher). S4 (LLM zero-shot) is read from a disk cache, added in M4.

## 4. Evaluation

Added in M1: accuracy, precision, recall and F1 (macro and weighted) for every system, mean and standard deviation over five seeds.

## 5. Router and cost

Added in M4: router policy, risk-coverage curves, coverage at matched risk, LLM calls and cost saved.

## 6. Limitations

Written in M6, once the results are in.